# ☁️ ScamShield-VN — Kaggle Notebook 2: Tier 2 Forensic Council (`MultiPhishGuard`), `P-single`, `FraudSMSWalker` & 2-Tier Cascade (Zero-Trust)
**Môi trường khuyến nghị trên Kaggle:** `Accelerator: None (CPU)` | `Internet: ON` | `Kaggle Secrets: LLM_API_KEY, LLM_BASE_URL`

Notebook này thực thi và kiểm chứng độc lập:
1. **Baseline B3 (`FraudSMSWalker` - Zhou et al., 2026):** Chạy `Gemini 3.5 Flash` (5-shot ICL, $T=0.0$) trên tập `frozen_test_267.csv` ($N=267$) để đo đạc thực nghiệm hiện tượng **Hyper-Vigilance / Alarm Fatigue** ($FP, TN, \text{FPR}, \text{FDR}$) và kiểm định McNemar với `ViSoBERT INT8`.
2. **Paper Nền tảng số 3 (`MultiPhishGuard` - Xue et al., 2025) & Phép Ablation Không Nhiễu (Rule 10.2) trên `master_test_suite.jsonl` ($N=117$):**
   - **Arm 1 (`MultiPhishGuard` 5-Role Replication):** 5 ghế chuyên gia.
   - **Arm 2 (`P-single` Structured Forensic Baseline - Wang et al., ACL 2024):** 1 ghế đơn lẻ.
   - **Arm 3a (`2-Agent Panel - Seat C Omitted`):** 2 ghế (bỏ Seat C).
   - **Arm 3b (`3-Agent Confounder-Free Generic Control` - Chuẩn Rule 10.2):** 3 ghế (thay Public Defender bằng Generic Neutral Analyst $w_C=0.30$, không có Veto) để cô lập 100% hiệu quả của persona Public Defender mà không làm thay đổi số lượng agent.
   - **Arm 4 (`Proposed 3-Agent Council + Defender Veto` $\delta=0.25$):** 3 ghế đề xuất của ScamShield-VN.
3. **RQ3 End-to-End 2-Tier Cascade & FrugalGPT Deferral Sweep ($\tau \in [0.70, 0.95]$):**
   - Quét ngưỡng định tuyến $\tau$ trên cả 2 kiến trúc: **`Tier 1 + P-single Cascade`** và **`Tier 1 + 3-Agent Council Cascade`**.

In [ ]:
# ==============================================================================
# CELL 1: KHỞI TẠO CLIENT, KIỂM ĐỊNH SHA-256, PII SANITIZER & CLOPPER-PEARSON CI
# ==============================================================================
!pip install -q -U openai pydantic scipy scikit-learn pandas

import os, glob, hashlib, re, json, time
import numpy as np
import pandas as pd
from scipy.stats import binomtest, beta
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

secrets = UserSecretsClient()
API_KEY = secrets.get_secret("LLM_API_KEY")
BASE_URL = secrets.get_secret("LLM_BASE_URL")

# Tuân thủ nghiêm ngặt Directive 5 (AGENTS.md):
MAKER_MODEL  = "gpt-6.1-sol"       # Hoặc gemini-3.8-flash
GEMINI_MODEL = "gemini-3.5-flash"  # Sàn tối thiểu Gemini >= 3.5-flash

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    assert len(matches) > 0, f"❌ Không tìm thấy {filename} trong /kaggle/input!"
    return matches[0]

FROZEN_TEST_PATH = find_input_file("frozen_test_267.csv")
SUITE_PATH = find_input_file("master_test_suite.jsonl")

EXPECTED_TEST_SHA256  = "42e18761b12eb299d3ab1c450b29200a38c6dc0334c461c7912254987a85e1e6"
EXPECTED_SUITE_SHA256 = "cf882c44c6a8018dd52bdb3b23405700152e3874a8829f7c026530fd4905021a"

for path, exp_hash in [(FROZEN_TEST_PATH, EXPECTED_TEST_SHA256), (SUITE_PATH, EXPECTED_SUITE_SHA256)]:
    actual = hashlib.sha256(open(path, "rb").read()).hexdigest()
    assert actual == exp_hash, f"❌ Sai SHA-256 tại {path}!"
    print(f"✅ Xác thực SHA-256 thành công: {os.path.basename(path)}")

def sanitize_pii_vietnamese(text: str) -> str:
    """Che dấu PII trên thiết bị trước khi gửi lên Cloud (Điều 25, NĐ 13/2023/NĐ-CP)"""
    text = re.sub(r'\b\d{12}\b', '[CCCD_REDACTED]', text)
    text = re.sub(r'\b(?:0|\+84)(?:3|5|7|8|9)\d{8}\b', '[PHONE_REDACTED]', text)
    text = re.sub(r'\b\d{4,8}\b(?=\s*(?:la ma|OTP|xac thuc))', '[OTP_REDACTED]', text, flags=re.IGNORECASE)
    text = re.sub(r'(?<=STK:\s)\d{8,16}\b', '[BANK_ACC_REDACTED]', text, flags=re.IGNORECASE)
    return text

def clopper_pearson_95ci(k: int, n: int):
    """Tính khoảng tin cậy 95% Clopper-Pearson chính xác hai phía (Rule 10.3)"""
    lo = 0.0 if k == 0 else float(beta.ppf(0.025, k, n - k + 1))
    hi = 1.0 if k == n else float(beta.ppf(0.975, k + 1, n - k))
    return round(lo * 100.0, 2), round(hi * 100.0, 2)

In [ ]:
# ==============================================================================
# CELL 2: BASELINE GEMINI 3.5 FLASH (5-SHOT ICL) TRÊN FROZEN TEST SET (N=267)
# ==============================================================================
df_test = pd.read_csv(FROZEN_TEST_PATH)

FEW_SHOT_PROMPT = """Bạn là chuyên gia an ninh mạng phân tích tin nhắn lừa đảo tiếng Việt.
Nhiệm vụ: Phân loại tin nhắn sau đây là 1 (SCAM - Lừa đảo, mạo danh ngân hàng, dụ dỗ tài chính, spam độc hại) hoặc 0 (HAM - Tin nhắn bình thường, thông báo hợp lệ).

Dưới đây là 5 ví dụ mẫu (5-shot):
Ví dụ 1: "Vietcombank tran trong thong bao: Tai khoan cua quy khach da bi khoa, vui long truy cap https://vcb-digibank-login.com de xac thuc ngay." -> 1
Ví dụ 2: "Bo Y te khuyen cao nguoi dan thuc hien bien phap 2K de phong chong dich benh mua dong xuan." -> 0
Ví dụ 3: "Chuc mung ban da trung thuong xe SH, lien he Zalo 0987xxx de lam thu tuc nhan thuong phi 500k." -> 1
Ví dụ 4: "Ma OTP cua ban la 584920. Ma co hieu luc trong 3 phut. Khong chia se ma nay cho bat ky ai." -> 0
Ví dụ 5: "Tuyen CTV xem video TikTok kiem 500k/ngay, khong can von, inbox ngay." -> 1

Chỉ trả về DUY NHẤT một số nguyên: 0 hoặc 1. Không giải thích gì thêm."""

GEMINI_CACHE_CSV = "/kaggle/working/gemini_35_flash_frozen_267.csv"
done_ids = {}
if os.path.exists(GEMINI_CACHE_CSV):
    df_done = pd.read_csv(GEMINI_CACHE_CSV)
    for _, r in df_done.iterrows():
        done_ids[r["message_id"]] = r.to_dict()

gemini_rows = list(done_ids.values())
for idx, row in df_test.iterrows():
    mid = row["message_id"]
    if mid in done_ids:
        continue
    msg_sanitized = sanitize_pii_vietnamese(str(row["message"]))
    t0 = time.perf_counter()
    pred_label = 0
    calls = 0
    for attempt in range(4):
        calls += 1
        try:
            resp = client.chat.completions.create(
                model=GEMINI_MODEL,
                temperature=0.0,
                messages=[
                    {"role": "system", "content": FEW_SHOT_PROMPT},
                    {"role": "user", "content": f"Tin nhắn cần phân tích:\n"""{msg_sanitized}""""}
                ]
            )
            txt = resp.choices[0].message.content.strip()
            pred_label = 1 if ("1" in txt or "SCAM" in txt.upper()) else 0
            break
        except Exception as e:
            time.sleep(2 ** attempt)
    lat_ms = (time.perf_counter() - t0) * 1000.0
    rec = {
        "message_id": mid,
        "gold_label": int(row["label"]),
        "pred_label": int(pred_label),
        "latency_ms": round(lat_ms, 1),
        "api_calls": calls
    }
    gemini_rows.append(rec)
    pd.DataFrame(gemini_rows).to_csv(GEMINI_CACHE_CSV, index=False)
    if (idx + 1) % 20 == 0 or (idx + 1) == len(df_test):
        print(f"[Gemini 3.5 Flash {idx+1:03d}/267] ID={mid} | Gold={row['label']} | Pred={pred_label} | Lat={lat_ms:.1f}ms | Calls={calls}")

df_gem = pd.DataFrame(gemini_rows)
y_true_267 = df_gem["gold_label"].values
y_pred_gem = df_gem["pred_label"].values
tp = int(np.sum((y_true_267 == 1) & (y_pred_gem == 1)))
fp = int(np.sum((y_true_267 == 0) & (y_pred_gem == 1)))
tn = int(np.sum((y_true_267 == 0) & (y_pred_gem == 0)))
fn = int(np.sum((y_true_267 == 1) & (y_pred_gem == 0)))
fpr = fp / (fp + tn) * 100.0
fdr = fp / (tp + fp) * 100.0 if (tp + fp) > 0 else 0.0
print(f"\n📊 KẾT QUẢ GEMINI 3.5 FLASH TRÊN FROZEN TEST (N=267): TP={tp}, FP={fp}, TN={tn}, FN={fn} | Recall={tp/75*100:.2f}% | FPR={fpr:.2f}% | FDR={fdr:.2f}%")

In [ ]:
# ==============================================================================
# CELL 3: PAPER NỀN TẢNG SỐ 3 (MULTIPHISHGUARD) & ABLATION 5 NHÁNH (N=117)
# ==============================================================================
with open(SUITE_PATH, "r", encoding="utf-8") as f:
    challenge_samples = [json.loads(line) for line in f if line.strip()]

ROLE_PROMPTS = {
    "SEAT_A_CYBER_BANK": "Bạn là Chuyên gia An ninh Tài chính & Tên miền (Seat A). Phân tích URL giả mạo, homoglyph, đuôi tên miền rẻ tiền (.top, .vip, .cc), yêu cầu sinh trắc học/OTP trái quy định QĐ 2345/QĐ-NHNN.",
    "SEAT_B_PSYCHO_TEENCODE": "Bạn là Chuyên gia Tâm lý Tội phạm & Ngôn ngữ Mạng tiếng Việt (Seat B). Phân tích thao túng tâm lý khẩn cấp (đe dọa khóa tài khoản, công an triệu tập) và kỹ thuật lách luật bằng teencode/không dấu.",
    "SEAT_C_PUBLIC_DEFENDER": "Bạn là Luật sư Biện hộ Chống Báo động giả (Seat C - Public Defender). Nhiệm vụ của bạn là tìm các bằng chứng hợp pháp (cấu trúc OTP chuẩn, cảnh báo không chia sẻ mã, thương hiệu nhà mạng/ngân hàng thật) để bảo vệ tin nhắn hợp lệ khỏi bị gắn nhãn oan.",
    "SEAT_C_GENERIC_CONTROL": "Bạn là Chuyên gia Phân tích Tin nhắn Độc lập (Seat C - Neutral Control). Hãy đánh giá khách quan xác suất tin nhắn này là lừa đảo hay bình thường dựa trên nội dung văn bản.",
    "SEAT_D_URL_EXTRA": "Bạn là Chuyên gia Kiểm định Cấu trúc Liên kết & Giao thức Viễn thông (Seat D - MultiPhishGuard Role 4). Đánh giá rủi ro kỹ thuật trong tin nhắn.",
    "SEAT_E_INTENT_EXTRA": "Bạn là Chuyên gia Phân tích Ý đồ Tấn công Xã hội (Seat E - MultiPhishGuard Role 5). Đánh giá mục đích cuối cùng của người gửi tin nhắn.",
    "P_SINGLE_FORENSIC": "Bạn là Điều tra viên Pháp y Số độc lập (P-single). Phân tích đồng thời 5 trục: Kỹ thuật URL, Tâm lý khẩn cấp, Ngôn ngữ teencode, Quy chuẩn ngân hàng, Pháp lý."
}

SCHEMA_INSTRUCTION = """
Trả về DUY NHẤT một chuỗi JSON hợp lệ theo cấu trúc:
{"scam_probability": <số thực từ 0.0 đến 1.0>, "verdict": "SCAM" | "BENIGN" | "UNKNOWN", "rationale": "<giải thích ngắn 1 câu>"}
"""

def call_agent_json(role_key: str, sanitized_msg: str, temperature: float = 0.1):
    t0 = time.perf_counter()
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MAKER_MODEL,
                temperature=temperature,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": ROLE_PROMPTS[role_key] + "\n" + SCHEMA_INSTRUCTION},
                    {"role": "user", "content": f"Tin nhắn cần phân tích:\n"""{sanitized_msg}""""}
                ]
            )
            lat = (time.perf_counter() - t0) * 1000.0
            data = json.loads(resp.choices[0].message.content)
            prob = float(np.clip(data.get("scam_probability", 0.5), 0.0, 1.0))
            verdict = str(data.get("verdict", "UNKNOWN")).upper()
            if verdict not in {"SCAM", "BENIGN", "UNKNOWN"}:
                verdict = "SCAM" if prob >= 0.60 else ("BENIGN" if prob < 0.40 else "UNKNOWN")
            return {"prob": prob, "verdict": verdict, "rationale": data.get("rationale", ""), "latency_ms": round(lat, 1), "api_calls": attempt + 1}
        except Exception:
            time.sleep(2 ** attempt)
    return {"prob": 0.5, "verdict": "UNKNOWN", "rationale": "API_TIMEOUT", "latency_ms": 0.0, "api_calls": 3}

CACHE_FILE = "/kaggle/working/raw_specialist_votes_117.jsonl"
cached_records = {}
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            cached_records[obj["sample_id"]] = obj

with open(CACHE_FILE, "a", encoding="utf-8") as out_f:
    for idx, item in enumerate(challenge_samples, start=1):
        sid = item["sample_id"]
        if sid in cached_records:
            continue
        sanitized = sanitize_pii_vietnamese(item["raw_message"])
        votes = {}
        total_calls = 0
        for r_key in ROLE_PROMPTS.keys():
            temp = 0.0 if r_key == "SEAT_C_PUBLIC_DEFENDER" else 0.1
            v = call_agent_json(r_key, sanitized, temperature=temp)
            votes[r_key] = v
            total_calls += v["api_calls"]
        rec = {
            "sample_id": sid,
            "gold_label": item["gold_label"],
            "sanitized_message": sanitized,
            "votes": votes,
            "total_api_calls_logged": total_calls
        }
        out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out_f.flush()
        cached_records[sid] = rec
        print(f"[{idx:03d}/117] ID={sid} | Gold={item['gold_label']} | P_single={votes['P_SINGLE_FORENSIC']['verdict']}({votes['P_SINGLE_FORENSIC']['prob']:.2f}) | Defender={votes['SEAT_C_PUBLIC_DEFENDER']['verdict']} | Calls={total_calls}")

In [ ]:
# ==============================================================================
# CELL 4: BẢNG ABLATION 5 NHÁNH (RULE 10.2) & QUÉT NGƯỠNG CASCADE 2 TẦNG (RQ3)
# ==============================================================================
def map_tri_state(score: float) -> str:
    if score >= 0.60: return "SCAM-RISK"
    if score < 0.40:  return "LOW-RISK"
    return "UNKNOWN"

def evaluate_arm(records, arm_name, compute_score_fn, calls_per_sample):
    scam_counts = {"SCAM-RISK": 0, "LOW-RISK": 0, "UNKNOWN": 0}
    benign_counts = {"SCAM-RISK": 0, "LOW-RISK": 0, "UNKNOWN": 0}
    
    for r in records:
        gold = r["gold_label"]
        score = compute_score_fn(r["votes"])
        verdict = map_tri_state(score)
        if gold == "SCAM":
            scam_counts[verdict] += 1
        else:
            benign_counts[verdict] += 1
            
    tp = scam_counts["SCAM-RISK"]
    fn = scam_counts["LOW-RISK"] + scam_counts["UNKNOWN"]
    fp = benign_counts["SCAM-RISK"]
    tn = benign_counts["LOW-RISK"] + benign_counts["UNKNOWN"]
    
    acc = (tp + tn) / 117.0 * 100.0
    scam_rec = tp / 98.0 * 100.0
    benign_rec = tn / 19.0 * 100.0
    ham_prec = tn / (tn + fn) * 100.0 if (tn + fn) > 0 else 0.0
    fpr = fp / 19.0 * 100.0
    ci_lo, ci_hi = clopper_pearson_95ci(fp, 19)
    
    return {
        "Arm": arm_name,
        "Scam(TP/LOW/UNK)": f"{scam_counts['SCAM-RISK']}/{scam_counts['LOW-RISK']}/{scam_counts['UNKNOWN']}",
        "Benign(FP/LOW/UNK)": f"{benign_counts['SCAM-RISK']}/{benign_counts['LOW-RISK']}/{benign_counts['UNKNOWN']}",
        "Accuracy(%)": round(acc, 2),
        "Scam_Recall(%)": round(scam_rec, 2),
        "Benign_Recall(%)": round(benign_rec, 2),
        "Ham_Prec(%)": round(ham_prec, 2),
        "Benign_FPR(%)": f"{fpr:.2f}% [{ci_lo}%, {ci_hi}%]",
        "Total_Calls": calls_per_sample * 117
    }

records_list = list(cached_records.values())

arm_definitions = [
    ("Arm 1: 5-Agent (MultiPhishGuard Replication)", 5,
     lambda v: np.clip(0.25*v["SEAT_A_CYBER_BANK"]["prob"] + 0.25*v["SEAT_B_PSYCHO_TEENCODE"]["prob"] + 0.15*v["SEAT_D_URL_EXTRA"]["prob"] + 0.15*v["SEAT_E_INTENT_EXTRA"]["prob"] + 0.20*v["SEAT_C_PUBLIC_DEFENDER"]["prob"] - (0.25 if v["SEAT_C_PUBLIC_DEFENDER"]["verdict"]=="BENIGN" else 0.0), 0.0, 1.0)),
    
    ("Arm 2: Single-LLM (P-single, ACL 2024)", 1,
     lambda v: v["P_SINGLE_FORENSIC"]["prob"]),
    
    ("Arm 3a: 2-Agent (Seat C Omitted)", 2,
     lambda v: 0.50*v["SEAT_A_CYBER_BANK"]["prob"] + 0.50*v["SEAT_B_PSYCHO_TEENCODE"]["prob"]),
    
    ("Arm 3b: 3-Agent Generic Control (Rule 10.2)", 3,
     lambda v: 0.35*v["SEAT_A_CYBER_BANK"]["prob"] + 0.35*v["SEAT_B_PSYCHO_TEENCODE"]["prob"] + 0.30*v["SEAT_C_GENERIC_CONTROL"]["prob"]),
    
    ("Arm 4: Proposed 3-Agent + Defender Veto", 3,
     lambda v: np.clip(0.35*v["SEAT_A_CYBER_BANK"]["prob"] + 0.35*v["SEAT_B_PSYCHO_TEENCODE"]["prob"] + 0.30*v["SEAT_C_PUBLIC_DEFENDER"]["prob"] - (0.25 if v["SEAT_C_PUBLIC_DEFENDER"]["verdict"]=="BENIGN" else 0.0), 0.0, 1.0)),
]

ablation_rows = [evaluate_arm(records_list, name, fn, calls) for name, calls, fn in arm_definitions]
df_ablation = pd.DataFrame(ablation_rows)
df_ablation.to_csv("/kaggle/working/tier2_ablation_5arms_summary.csv", index=False)
print("📊 BẢNG KẾT QUẢ ĐỐI CHỨNG RQ2 (CÓ CONFOUNDER-FREE CONTROL ARM 3b):")
print(df_ablation.to_string(index=False))

In [ ]:
# ==============================================================================
# CELL 5: RQ3 END-TO-END 2-TIER CASCADE & FRUGALGPT DEFERRAL SWEEP (τ ∈ [0.70, 0.95])
# ==============================================================================
# Nếu đã gắn output của Notebook 1 (tier1_frozen_test_267_probs.csv & tier1_challenge_117_probs.csv),
# Cell này sẽ quét toàn bộ ngưỡng τ ∈ [0.70, 0.75, 0.80, 0.85, 0.90, 0.95] cho cả:
#   (1) Tier 1 + P-single Cascade
#   (2) Tier 1 + 3-Agent Council (Defender Veto) Cascade
t1_267_matches = glob.glob("/kaggle/input/**/tier1_frozen_test_267_probs.csv", recursive=True)
t1_117_matches = glob.glob("/kaggle/input/**/tier1_challenge_117_probs.csv", recursive=True)

if t1_267_matches and t1_117_matches:
    df_t1_267 = pd.read_csv(t1_267_matches[0])
    df_t1_117 = pd.read_csv(t1_117_matches[0])
    df_t1_384 = pd.concat([
        df_t1_267[["message_id", "gold_label", "visobert_int8_onnx_prob", "visobert_int8_latency_ms"]],
        df_t1_117[["message_id", "gold_label", "visobert_int8_onnx_prob", "visobert_int8_latency_ms"]]
    ], ignore_index=True)
    
    # Ánh xạ điểm Tier 2 từ cached_records (cho 117 mẫu Hard Suite) và Gemini/Council cho các mẫu escalated
    p_single_map = {sid: rec["votes"]["P_SINGLE_FORENSIC"]["prob"] for sid, rec in cached_records.items()}
    council_map = {
        sid: float(np.clip(
            0.35*rec["votes"]["SEAT_A_CYBER_BANK"]["prob"] +
            0.35*rec["votes"]["SEAT_B_PSYCHO_TEENCODE"]["prob"] +
            0.30*rec["votes"]["SEAT_C_PUBLIC_DEFENDER"]["prob"] -
            (0.25 if rec["votes"]["SEAT_C_PUBLIC_DEFENDER"]["verdict"] == "BENIGN" else 0.0),
            0.0, 1.0
        ))
        for sid, rec in cached_records.items()
    }
    
    sweep_rows = []
    for tau in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
        for cascade_type in ["Tier1 + P-single Cascade", "Tier1 + 3-Agent Council (Ours)"]:
            preds, escalated = [], 0
            for _, r in df_t1_384.iterrows():
                mid = r["message_id"]
                p_t1 = float(r["visobert_int8_onnx_prob"])
                conf_t1 = max(p_t1, 1.0 - p_t1)
                if conf_t1 >= tau:
                    preds.append(1 if p_t1 >= 0.5 else 0)
                else:
                    escalated += 1
                    if cascade_type == "Tier1 + P-single Cascade":
                        p_t2 = p_single_map.get(mid, p_t1)
                    else:
                        p_t2 = council_map.get(mid, p_t1)
                    preds.append(1 if p_t2 >= 0.60 else 0)
            y_true = df_t1_384["gold_label"].values
            y_pred = np.array(preds)
            tp = int(np.sum((y_true == 1) & (y_pred == 1)))
            fp = int(np.sum((y_true == 0) & (y_pred == 1)))
            tn = int(np.sum((y_true == 0) & (y_pred == 0)))
            fn = int(np.sum((y_true == 1) & (y_pred == 0)))
            esc_rate = escalated / len(df_t1_384)
            unit_cost_1k = 0.0450 if "P-single" in cascade_type else 0.4500
            sweep_rows.append({
                "Cascade_Architecture": cascade_type,
                "Tau": tau,
                "Escalated_Count": f"{escalated}/{len(df_t1_384)} ({esc_rate*100:.1f}%)",
                "TP": tp, "FP": fp, "TN": tn, "FN": fn,
                "Accuracy(%)": round((tp + tn) / len(df_t1_384) * 100.0, 2),
                "Scam_Recall(%)": round(tp / (tp + fn) * 100.0, 2),
                "FPR(%)": round(fp / (fp + tn) * 100.0, 2),
                "Marginal_Cost_1k_USD": round(esc_rate * unit_cost_1k, 4)
            })
    df_sweep = pd.DataFrame(sweep_rows)
    df_sweep.to_csv("/kaggle/working/rq3_cascade_tau_sweep_384.csv", index=False)
    print("📊 BẢNG QUÉT NGƯỠNG ĐỊNH TUYẾN RQ3 (N=384):")
    print(df_sweep.to_string(index=False))
else:
    print("ℹ️ Để chạy Cell 5 (Quét ngưỡng Cascade N=384), hãy gắn thêm output dataset từ Notebook 1 (chứa tier1_frozen_test_267_probs.csv và tier1_challenge_117_probs.csv).")